> **Superseded.** This notebook is kept only as a Colab-runnable reference for the original pipeline.
>
> The source of truth is now the `ecg/` package and `scripts/train.py`, which run locally, are
> unit-tested, and fix several evaluation issues this notebook had (threshold tuned on test instead
> of validation, normal-beat cap removed on val/test, only the float model evaluated, unshuffled
> quantization calibration set). See `README.md` for how to run the package version.


In [8]:
# ==========================================
# 0. COLAB ENVIRONMENT SETUP
# ==========================================
!pip install -q wfdb

import os
import numpy as np
import scipy.signal as signal
import wfdb
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from google.colab import files

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# ==========================================
# 1. DATA ACQUISITION & ENRICHED PROCESSING
# ==========================================
train_records = ['101', '106', '108', '112', '114', '115', '118', '119', '200', '201', '203', '205', '208', '209', '210', '212', '213', '215', '219', '221', '222', '223']
val_records   = ['103', '109', '111', '124', '220', '230', '231', '232']
test_records  = ['100', '105', '113', '121', '123', '202', '214', '233']

# Binary mapping: 0 = Normal, 1 = Abnormal
symbol_map = {
    'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0,
    'V': 1, 'E': 1,
    'A': 1, 'a': 1, 'J': 1, 'S': 1
}

def load_dual_input_spectrogram_data(record_list, max_normal_per_rec=600, augment=False):
    X_spec_list, X_rr_list, y_list = [], [], []
    WINDOW_SIZE = 250
    HALF_WINDOW = WINDOW_SIZE // 2

    for rec in record_list:
        try:
            record = wfdb.rdrecord(rec, pn_dir='mitdb')
            annotation = wfdb.rdann(rec, pn_dir='mitdb', extension='atr')
        except ValueError:
            continue

        fs = record.fs
        nyq = 0.5 * fs
        b, a = signal.butter(2, [0.5/nyq, 40.0/nyq], btype='band')
        filtered = signal.filtfilt(b, a, record.p_signal[:, 0])

        samples = annotation.sample
        symbols = annotation.symbol
        rr_intervals = np.diff(samples) / fs
        normal_count = 0

        for i in range(1, len(samples) - 1):
            sample_idx = samples[i]
            symbol = symbols[i]

            if symbol in symbol_map and (sample_idx - HALF_WINDOW >= 0) and (sample_idx + HALF_WINDOW < len(filtered)):
                target_class = symbol_map[symbol]

                if target_class == 0:
                    if normal_count >= max_normal_per_rec:
                        continue
                    normal_count += 1

                # Data Augmentation: Simulate R-peak detection jitter during training
                jitter_offsets = [0, np.random.randint(-8, 9)] if augment else [0]

                for jitter in jitter_offsets:
                    adj_idx = sample_idx + jitter
                    if (adj_idx - HALF_WINDOW < 0) or (adj_idx + HALF_WINDOW >= len(filtered)):
                        adj_idx = sample_idx

                    window = filtered[adj_idx - HALF_WINDOW : adj_idx + HALF_WINDOW]

                    # Higher Resolution STFT Spectrogram (nperseg=64, noverlap=48)
                    f, t, Sxx = signal.spectrogram(window, fs=fs, nperseg=64, noverlap=48)
                    Sxx_log = np.log1p(Sxx * 1000)
                    s_min, s_max = np.min(Sxx_log), np.max(Sxx_log)
                    norm_spec = (Sxx_log - s_min) / (s_max - s_min) if s_max - s_min > 0 else Sxx_log

                    # Extract 5 timing features (Includes pre_rr / post_rr ratio)
                    pre_rr = rr_intervals[i - 1]
                    post_rr = rr_intervals[i]
                    start_k = max(0, i - 10)
                    local_mean_rr = np.mean(rr_intervals[start_k:i])
                    rr_ratio = pre_rr / local_mean_rr if local_mean_rr > 0 else 1.0
                    pre_post_ratio = pre_rr / post_rr if post_rr > 0 else 1.0

                    X_spec_list.append(norm_spec)
                    X_rr_list.append([pre_rr, post_rr, local_mean_rr, rr_ratio, pre_post_ratio])
                    y_list.append(target_class)

    return (np.array(X_spec_list, dtype=np.float32)[..., np.newaxis],
            np.array(X_rr_list, dtype=np.float32),
            np.array(y_list, dtype=np.int32))

print("\nExtracting High-Resolution STFT & 5-Feature R-R Timing Data...")
X_train_spec, X_train_rr, y_train = load_dual_input_spectrogram_data(train_records, max_normal_per_rec=600, augment=True)
X_val_spec, X_val_rr, y_val       = load_dual_input_spectrogram_data(val_records, max_normal_per_rec=600, augment=False)
X_test_spec, X_test_rr, y_test    = load_dual_input_spectrogram_data(test_records, max_normal_per_rec=600, augment=False)

print(f"Train Set: {len(X_train_spec)} | Val Set: {len(X_val_spec)} | Test Set: {len(X_test_spec)}")

spec_shape = X_train_spec.shape[1:]

# ==========================================
# 2. MULTI-SCALE MODEL ARCHITECTURE
# ==========================================
def build_multiscale_ptq_model(spec_shape):
    spec_input = tf.keras.layers.Input(shape=spec_shape, name="spectrogram_input")

    # Parallel Multi-Scale Kernel Branches
    branch_temporal = tf.keras.layers.Conv2D(filters=8, kernel_size=(1, 5), padding='same', activation='relu6', kernel_regularizer='l2')(spec_input)
    branch_spectral = tf.keras.layers.Conv2D(filters=8, kernel_size=(5, 1), padding='same', activation='relu6', kernel_regularizer='l2')(spec_input)

    x = tf.keras.layers.concatenate([branch_temporal, branch_spectral])
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool2D(pool_size=(2, 2))(x)

    x = tf.keras.layers.Conv2D(filters=32, kernel_size=(3, 3), padding='same', activation='relu6', kernel_regularizer='l2')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Flatten()(x)

    # 5-Feature R-R Timing Branch
    rr_input = tf.keras.layers.Input(shape=(5,), name="rr_input")
    y = tf.keras.layers.Dense(8, activation='relu6')(rr_input)

    # Dense Fusion Network
    combined = tf.keras.layers.concatenate([x, y])
    z = tf.keras.layers.Dense(64, activation='relu6', kernel_regularizer='l2')(combined)
    z = tf.keras.layers.Dropout(0.40)(z)
    output = tf.keras.layers.Dense(1, activation='sigmoid', name="output")(z)

    return tf.keras.models.Model(inputs=[spec_input, rr_input], outputs=output)

# Custom Focal Loss Function
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce = - (y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        focal_factor = tf.pow(1.0 - p_t, gamma)
        alpha_factor = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)
        return tf.reduce_mean(alpha_factor * focal_factor * bce)
    return loss

model = build_multiscale_ptq_model(spec_shape)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=['accuracy']
)

model.summary()

# ==========================================
# 3. TRAINING PIPELINE
# ==========================================
early_stop_cb = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)

print("\nStarting Training with Focal Loss & Data Augmentation...")
history = model.fit(
    x=[X_train_spec, X_train_rr],
    y=y_train,
    validation_data=([X_val_spec, X_val_rr], y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop_cb]
)

# ==========================================
# 4. EVALUATION
# ==========================================
print("\n--- Model Evaluation on Unseen Test Patients ---")
y_pred_probs = model.predict([X_test_spec, X_test_rr])

# Evaluate across multiple decision thresholds
thresholds = [0.50, 0.55, 0.60]
for t in thresholds:
    y_pred_adj = (y_pred_probs > t).astype(int).flatten()
    print(f"\n=== THRESHOLD: {t:.2f} ===")
    print(classification_report(y_test, y_pred_adj, target_names=['Normal (0)', 'Abnormal (1)'], zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_adj))

# ==========================================
# 5. POST-TRAINING QUANTIZATION (PTQ) & EXPORT
# ==========================================
def representative_data_gen():
    for i in range(100):
        yield {
            "spectrogram_input": X_train_spec[i:i+1].astype(np.float32),
            "rr_input": X_train_rr[i:i+1].astype(np.float32)
        }

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_int8_model = converter.convert()

header_filename = "model_data.h"
with open(header_filename, "w") as f:
    f.write("#ifndef MODEL_DATA_H_\n#define MODEL_DATA_H_\n\n")
    f.write(f"const unsigned int g_model_len = {len(tflite_int8_model)};\n")
    f.write("const unsigned char g_model[] __attribute__((aligned(4))) = {\n  ")
    for i, byte in enumerate(tflite_int8_model):
        f.write(f"0x{byte:02x}, ")
        if (i + 1) % 12 == 0:
            f.write("\n  ")
    f.write("\n};\n\n#endif  // MODEL_DATA_H_\n")

print(f"\nModel converted via PTQ! Final INT8 size: {len(tflite_int8_model) / 1024:.2f} KB")

# Download header file directly to PC's Downloads folder
files.download(header_filename)

TensorFlow Version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Extracting High-Resolution STFT & 5-Feature R-R Timing Data...
Train Set: 38582 | Val Set: 6196 | Test Set: 6053
Model: "model_5"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 spectrogram_input (InputLa  [(None, 33, 12, 1)]          0         []                            
 yer)                                                                                             
                                                                                                  
 conv2d_10 (Conv2D)          (None, 33, 12, 8)            48        ['spectrogram_input[0][0]']   
                                                                                                  
 conv2d_11 (Conv2D)          (None, 33, 12, 8)            48       

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



Model converted via PTQ! Final INT8 size: 206.95 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# Sweep lower thresholds to boost Abnormal Recall
lower_thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]

for t in lower_thresholds:
    y_pred_adj = (y_pred_probs > t).astype(int).flatten()
    cm = confusion_matrix(y_test, y_pred_adj)

    print(f"=== THRESHOLD: {t:.2f} ===")
    print(f"False Alarms (Normal flagged as Abnormal): {cm[0][1]}")
    print(f"Missed Abnormalities (Abnormal flagged as Normal): {cm[1][0]}")
    print(classification_report(y_test, y_pred_adj, target_names=['Normal (0)', 'Abnormal (1)'], zero_division=0))
    print("-" * 50)

=== THRESHOLD: 0.30 ===
False Alarms (Normal flagged as Abnormal): 95
Missed Abnormalities (Abnormal flagged as Normal): 69
              precision    recall  f1-score   support

  Normal (0)       0.99      0.98      0.98      4800
Abnormal (1)       0.93      0.94      0.94      1253

    accuracy                           0.97      6053
   macro avg       0.96      0.96      0.96      6053
weighted avg       0.97      0.97      0.97      6053

--------------------------------------------------
=== THRESHOLD: 0.35 ===
False Alarms (Normal flagged as Abnormal): 59
Missed Abnormalities (Abnormal flagged as Normal): 112
              precision    recall  f1-score   support

  Normal (0)       0.98      0.99      0.98      4800
Abnormal (1)       0.95      0.91      0.93      1253

    accuracy                           0.97      6053
   macro avg       0.96      0.95      0.96      6053
weighted avg       0.97      0.97      0.97      6053

----------------------------------------------

Threshold 0.30 is most optimal

In [10]:
from google.colab import files

# 1. Re-trigger download of the C Header file for Arduino / ESP32
files.download('model_data.h')

# 2. Save and download the raw INT8 .tflite binary
with open("model_int8.tflite", "wb") as f:
    f.write(tflite_int8_model)

files.download('model_int8.tflite')

# 3. Save and download the high-precision Float32 Keras model (.keras)
model.save("ecg_multiscale_model.keras")
files.download("ecg_multiscale_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>